# 05 — MCP Agent Deployment

**Purpose:** Deploy the MCP tool-calling agent as its own Mosaic AI Model Serving
endpoint — a real, reusable, chattable service, not a scripted notebook demo. The agent
discovers all 10 `agent_tools` functions over the real MCP protocol and routes each request
to the right one: answer a question via RAG, walk the user through the report menu and build
whichever one they pick, or classify a new applicant.

### What This Notebook Does
1. Defines an `mlflow.pyfunc.ChatAgent` that connects to the managed MCP endpoint from
   `src/04_agent_tools_mcp_functions`, discovers the tools, and runs a tool-calling loop. A
   system prompt tells it to call `list_report_types` and let the user choose before it ever
   calls a specific report generator, instead of guessing which one they meant.
2. Smoke-tests the agent locally on 4 demo prompts (a RAG question, a report-menu request, a
   specific report request, and a classification).
3. Logs, registers, and deploys it to a Mosaic AI Model Serving endpoint, then queries the
   deployed service live to prove it works end-to-end.

### Source & Target
| Direction | Resource |
|-----------|----------|
| Source | Managed MCP endpoint `/api/2.0/mcp/functions/insurance_rag_agent/agent_tools` |
| Target | UC model `insurance_rag_agent.agent_tools.insurance_mcp_agent` + serving endpoint |

> **Prerequisites:** Run `src/04_agent_tools_mcp_functions` first. Deploying consumes Free
> Edition's limited model-serving quota.

---

In [ ]:
%pip install -U mlflow databricks-agents databricks-mcp mcp "databricks-sdk[openai]" openai
dbutils.library.restartPython()

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG               = 'insurance_rag_agent'
AGENT_TOOLS_SCHEMA     = 'agent_tools'
# Get CHAT_MODEL_ENDPOINT and MODEL_QUERY_BASE_PATH from Serving > (your chat model) >
# "Use this model" > the model= and base_url= values in the generated code snippet — see
# docs/SETUP.md step 7 for the two naming patterns we've seen (AI Gateway vs. classic).
CHAT_MODEL_ENDPOINT    = f'{CATALOG}.agent_tools.meta-llama-3-3-70b-instruct_backend'
MODEL_QUERY_BASE_PATH  = '/ai-gateway/mlflow/v1'
AGENT_MODEL_NAME       = f'{CATALOG}.agent_tools.insurance_mcp_agent'
AGENT_ENDPOINT_NAME    = 'insurance_mcp_agent_endpoint'

# Keeps the agent from guessing which report the user wants — it has to show the menu first.
SYSTEM_PROMPT = (
    'You are an insurance data analysis assistant with three kinds of tools: answering '
    'questions about the policy data, generating written analysis reports, and classifying '
    'new applicants into a cost-risk tier. If someone asks for "a report" or "an analysis" '
    "without naming a specific one, call list_report_types first, show them the numbered "
    'options in your reply, and wait for them to pick one — do not guess which report they '
    'want or call more than one report generator per request.'
)

DEMO_PROMPTS = [
    'What do the records show about smokers in the southeast region with high charges?',
    'What kinds of reports can you generate for me?',
    'Generate the smoker comparison report.',
    'Classify this applicant: 52-year-old smoker, BMI 34, from the southeast, with 2 children.',
]

print(f'Chat model endpoint: {CHAT_MODEL_ENDPOINT}')
print(f'Registered model: {AGENT_MODEL_NAME}')
print(f'Demo prompts: {len(DEMO_PROMPTS)}')

In [ ]:
import json
import uuid

import mlflow
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse

mlflow.set_registry_uri('databricks-uc')

---

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

w              = WorkspaceClient()
mcp_server_url = f'{w.config.host}/api/2.0/mcp/functions/{CATALOG}/{AGENT_TOOLS_SCHEMA}'

mcp_client       = DatabricksMCPClient(server_url=mcp_server_url, workspace_client=w)
discovered_tools = mcp_client.list_tools()

for tool in discovered_tools:
    print(f'- {tool.name}: {tool.description}')

---

In [ ]:
class InsuranceMCPAgent(ChatAgent):
    '''Discovers the agent_tools MCP functions and routes each request to the right one via
    tool-calling. Clients are created inside predict() rather than __init__ so the agent
    instance stays picklable for MLflow logging.'''

    def __init__(self, mcp_server_url, chat_model_endpoint, base_path, system_prompt):
        self.mcp_server_url      = mcp_server_url
        self.chat_model_endpoint = chat_model_endpoint
        self.base_path           = base_path
        self.system_prompt       = system_prompt

    def predict(self, messages, context=None, custom_inputs=None):
        from databricks.sdk import WorkspaceClient
        from databricks_mcp import DatabricksMCPClient
        from openai import OpenAI

        w          = WorkspaceClient()
        mcp_client = DatabricksMCPClient(server_url=self.mcp_server_url, workspace_client=w)

        openai_tools = [
            {
                'type': 'function',
                'function': {
                    'name':        tool.name,
                    'description': tool.description,
                    'parameters':  tool.inputSchema,
                },
            }
            for tool in mcp_client.list_tools()
        ]

        # See notebook 03 for why this stays a manually-built OpenAI client rather than
        # w.serving_endpoints.get_open_ai_client() — CHAT_MODEL_ENDPOINT lives behind this
        # workspace's AI Gateway path, not the classic /serving-endpoints path. It's backed by
        # a scoped credential regardless: log_model() below declares the MCP server and the
        # chat model as `resources`, so Databricks injects a managed credential for both into
        # the deployed container.
        bearer_token  = w.config.authenticate()['Authorization'].split(' ', 1)[1]
        openai_client = OpenAI(
            api_key  = bearer_token,
            base_url = f'{w.config.host}{self.base_path}',
        )

        chat_messages = (
            [{'role': 'system', 'content': self.system_prompt}]
            + [{'role': m.role, 'content': m.content} for m in messages]
        )

        first_response = openai_client.chat.completions.create(
            model    = self.chat_model_endpoint,
            messages = chat_messages,
            tools    = openai_tools,
        )
        choice = first_response.choices[0].message

        if not choice.tool_calls:
            return ChatAgentResponse(messages=[
                ChatAgentMessage(id=str(uuid.uuid4()), role='assistant', content=choice.content)
            ])

        chat_messages.append({
            'role':       'assistant',
            'content':    choice.content,
            'tool_calls': [
                {
                    'id':       tc.id,
                    'type':     'function',
                    'function': {'name': tc.function.name, 'arguments': tc.function.arguments},
                }
                for tc in choice.tool_calls
            ],
        })

        for tool_call in choice.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            tool_result = mcp_client.call_tool(tool_name, tool_args)
            result_text = tool_result.content[0].text

            chat_messages.append({
                'role':         'tool',
                'tool_call_id': tool_call.id,
                'content':      result_text,
            })

        final_response = openai_client.chat.completions.create(
            model    = self.chat_model_endpoint,
            messages = chat_messages,
        )
        answer = final_response.choices[0].message.content

        return ChatAgentResponse(messages=[
            ChatAgentMessage(id=str(uuid.uuid4()), role='assistant', content=answer)
        ])


mcp_agent = InsuranceMCPAgent(
    mcp_server_url      = mcp_server_url,
    chat_model_endpoint = CHAT_MODEL_ENDPOINT,
    base_path           = MODEL_QUERY_BASE_PATH,
    system_prompt       = SYSTEM_PROMPT,
)

for prompt in DEMO_PROMPTS:
    smoke_test = mcp_agent.predict(
        [ChatAgentMessage(id=str(uuid.uuid4()), role='user', content=prompt)]
    )
    print(f'\n=== User: {prompt} ===')
    print(f'Assistant: {smoke_test.messages[-1].content}')

---

In [ ]:
from mlflow.models.resources import DatabricksServingEndpoint

# get_databricks_resources() resolves the underlying UC functions behind the MCP server into
# the resource objects log_model() needs — no DatabricksMCPServer resource class exists.
mcp_resources = DatabricksMCPClient().get_databricks_resources(mcp_server_url)
resources     = mcp_resources + [DatabricksServingEndpoint(endpoint_name=CHAT_MODEL_ENDPOINT)]

# Pin the served container's requirements to exactly what predict() imports, instead of
# letting log_model() auto-capture this whole notebook's environment. The %pip install cell
# above installs the *full* mlflow package (not mlflow-skinny), which drags in scikit-learn,
# matplotlib, scipy, boto3, google-cloud-storage, docker, gunicorn, flask, and more — none of
# which predict() uses. Auto-capturing all of that makes the serving container build large,
# slow, and more likely to time out or fail; this keeps the build to what's actually needed.
pip_requirements = [
    'mlflow',
    'databricks-mcp',
    'mcp',
    'databricks-sdk[openai]',
    'openai',
]

with mlflow.start_run(run_name='insurance_mcp_agent'):
    logged_model = mlflow.pyfunc.log_model(
        name                  = 'mcp_agent',
        python_model          = mcp_agent,
        registered_model_name = AGENT_MODEL_NAME,
        resources             = resources,
        pip_requirements      = pip_requirements,
    )

model_version = logged_model.registered_model_version
print(f'Registered {AGENT_MODEL_NAME} version {model_version}')

---

In [ ]:
from databricks import agents
from mlflow import MlflowClient

# Deploy whatever is actually newest in Unity Catalog, not whatever `model_version` happens
# to hold in this session — if the logging cell above wasn't the last thing run before this
# one (e.g. after a Python restart), model_version could be stale or undefined, and this
# guarantees the deploy targets the real latest version regardless.
latest_version = max(
    int(mv.version) for mv in MlflowClient().search_model_versions(f"name='{AGENT_MODEL_NAME}'")
)
print(f'Latest registered version: {latest_version}')

deployment_info = agents.deploy(
    model_name    = AGENT_MODEL_NAME,
    model_version = latest_version,
    endpoint_name = AGENT_ENDPOINT_NAME,
    scale_to_zero = True,  # required on Free Edition — no provisioned/always-on throughput
)

print(f'Deployed endpoint: {deployment_info.endpoint_name}')

---

In [ ]:
# Uses AGENT_ENDPOINT_NAME (a static config constant) rather than deployment_info.endpoint_name,
# so this cell works even in a fresh session where the deploy cell above hasn't been re-run.
serving_client = w.serving_endpoints.get_open_ai_client()

for prompt in DEMO_PROMPTS:
    live_response = serving_client.chat.completions.create(
        model    = AGENT_ENDPOINT_NAME,
        messages = [{'role': 'user', 'content': prompt}],
    )
    print(f'\n=== User: {prompt} ===')
    print(f'Assistant: {live_response.choices[0].message.content}')